# North-Star HF Runner (Kaggle GPU)

This notebook is a Kaggle-first runner for the full HF north-star path:
1. Authenticate runtime
2. Pull the finetuned adapter from Kaggle dataset
3. Run base + finetuned multi-seed benchmark
4. Compute delta
5. Run transfer matrix

Use this in a Kaggle GPU notebook to avoid heavy local runs.

In [1]:
!nvidia-smi

Thu Apr  2 14:16:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             28W /   70W |    3437MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
from getpass import getpass
from pathlib import Path

REPO_NAME = 'tool-calling-reliability-benchmark'
REPO_URL = 'https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git'

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    kaggle_repo = Path('/kaggle/working') / REPO_NAME
    if not kaggle_repo.exists():
        print(f'[setup] Cloning repo to {kaggle_repo} ...')
        subprocess.run(['git', 'clone', REPO_URL, str(kaggle_repo)], check=True)
    repo_root = kaggle_repo

REPO_ROOT = repo_root.resolve()
os.chdir(REPO_ROOT)

if shutil.which('uv') is None:
    print('[setup] Installing uv ...')
    subprocess.run(['python', '-m', 'pip', 'install', '-q', 'uv'], check=True)

print('Repo root:', REPO_ROOT)
print('Kernel cwd:', Path.cwd())


Repo root: /kaggle/working/tool-calling-reliability-benchmark
Kernel cwd: /kaggle/working/tool-calling-reliability-benchmark


In [3]:
# Runtime auth (prompt based).
HF_TOKEN = str(os.environ.get('HF_TOKEN', '')).strip()
if not HF_TOKEN:
    HF_TOKEN = getpass('Enter HF_TOKEN (input hidden): ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required.')
os.environ['HF_TOKEN'] = HF_TOKEN

KAGGLE_USERNAME = str(os.environ.get('KAGGLE_USERNAME', '')).strip()
if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = input('Enter KAGGLE_USERNAME: ').strip()

KAGGLE_KEY = str(os.environ.get('KAGGLE_KEY', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = str(os.environ.get('KAGGLE_API_TOKEN', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = getpass('Enter KAGGLE_KEY (input hidden): ').strip()

if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise RuntimeError('KAGGLE_USERNAME and KAGGLE_KEY are required.')

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

try:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('[auth] Hugging Face login succeeded.')
except Exception as exc:
    print('[auth] HF login warning:', exc)

print('[auth] Kaggle credentials configured for runtime.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[auth] Hugging Face login succeeded.
[auth] Kaggle credentials configured for runtime.


In [5]:
# Pull latest adapter artifact into local repo tree in the Kaggle runtime.
DATASET = 'aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts'

# Keep Kaggle clone aligned with current repo so helper scripts exist.
sync_cmd = ['git', 'pull', '--ff-only', 'origin', 'main']
print('Running:', ' '.join(sync_cmd))
sync_res = subprocess.run(sync_cmd, text=True, capture_output=True, check=False)
if sync_res.stdout:
    print(sync_res.stdout)
if sync_res.returncode != 0:
    if sync_res.stderr:
        print(sync_res.stderr)
    raise RuntimeError(f'Git sync failed with code {sync_res.returncode}')

# Some Kaggle images emit sitecustomize warnings when wrapt is absent.
wrapt_install = ['uv', 'pip', 'install', '--python', '.venv/bin/python', 'wrapt']
subprocess.run(wrapt_install, text=True, capture_output=True, check=False)

pull_script = REPO_ROOT / 'scripts' / 'pull_kaggle_adapter.py'
if pull_script.exists():
    pull_cmd = [
        'uv', 'run', 'python', str(pull_script),
        '--dataset', DATASET,
        '--repo-root', '.',
    ]
    print('Running:', ' '.join(pull_cmd))
    res = subprocess.run(pull_cmd, text=True, capture_output=True, check=False)
    if res.stdout:
        print(res.stdout)
    if res.returncode != 0:
        if res.stderr:
            print(res.stderr)
        raise RuntimeError(f'Adapter pull failed with code {res.returncode}')
else:
    print('[adapter] pull_kaggle_adapter.py missing after sync; using direct Kaggle fallback.')
    download_dir = REPO_ROOT / 'tmp' / 'kaggle_adapter_pull'
    if download_dir.exists():
        shutil.rmtree(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    dl_cmd = [
        'kaggle', 'datasets', 'download',
        '-d', DATASET,
        '-p', str(download_dir),
        '--unzip', '-o', '-q',
    ]
    print('Running:', ' '.join(dl_cmd))
    dl_res = subprocess.run(dl_cmd, text=True, capture_output=True, check=False)
    if dl_res.stdout:
        print(dl_res.stdout)
    if dl_res.returncode != 0:
        if dl_res.stderr:
            print(dl_res.stderr)
        raise RuntimeError(f'Kaggle direct download failed with code {dl_res.returncode}')

    src = download_dir / 'adapter'
    dst = REPO_ROOT / 'outputs' / 'ft-notebook' / 'final'
    if not src.exists():
        raise FileNotFoundError(f'Missing adapter dir in downloaded payload: {src}')
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print('[adapter] Copied adapter payload to:', dst)

adapter_cfg = REPO_ROOT / 'outputs' / 'ft-notebook' / 'final' / 'adapter_config.json'
if not adapter_cfg.exists():
    raise FileNotFoundError(f'Missing adapter config: {adapter_cfg}')

cfg = json.loads(adapter_cfg.read_text(encoding='utf-8'))
print('[adapter] base_model_name_or_path =', cfg.get('base_model_name_or_path'))
print('[adapter] target_modules count =', len(cfg.get('target_modules', [])))

Running: git pull --ff-only origin main
Updating 6b37d38..627bd83
Fast-forward
 README.md                                |  15 +
 analysis/finetuning_entrypoint.ipynb     | 909 ++++++++++++++++++++++++++-----
 configs/planners/hf_qwen2_5_3b_base.json |   5 +
 configs/planners/hf_qwen2_5_3b_ft.json   |   6 +
 scripts/pull_kaggle_adapter.py           | 124 +++++
 scripts/run_northstar_hf.py              | 213 ++++++++
 src/tcrb/hf_planner.py                   | 241 ++++++++
 src/tcrb/planner.py                      |  55 ++
 8 files changed, 1434 insertions(+), 134 deletions(-)
 create mode 100644 configs/planners/hf_qwen2_5_3b_base.json
 create mode 100644 configs/planners/hf_qwen2_5_3b_ft.json
 create mode 100644 scripts/pull_kaggle_adapter.py
 create mode 100644 scripts/run_northstar_hf.py
 create mode 100644 src/tcrb/hf_planner.py

Running: uv run python /kaggle/working/tool-calling-reliability-benchmark/scripts/pull_kaggle_adapter.py --dataset aaliyanshaikh/tcrb-qwen25-3b-adapter-ar

In [9]:
# Ensure benchmark/runtime dependencies exist in the repo's uv environment.
required_mods = ['torch', 'transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes']

# Verify modules in uv-managed environment (not only notebook kernel env).
probe = [
    'uv', 'run', 'python', '-c',
    "import importlib.util as u; mods=%r; missing=[m for m in mods if u.find_spec(m) is None]; print('MISSING=' + ','.join(missing))" % required_mods,
 ]
probe_res = subprocess.run(probe, text=True, capture_output=True, check=False)
probe_out = (probe_res.stdout or '').strip()
print('[deps] Probe output:', probe_out)

missing = []
if 'MISSING=' in probe_out:
    missing_text = probe_out.split('MISSING=', 1)[1].strip()
    if missing_text:
        missing = [m for m in missing_text.split(',') if m]

if missing:
    # Install directly into uv venv interpreter used by `uv run`.
    install_cmd = [
        'uv', 'pip', 'install', '--python', '.venv/bin/python',
        'torch', 'transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes', 'wrapt',
    ]
    print('Running:', ' '.join(install_cmd))
    install_res = subprocess.run(install_cmd, text=True, capture_output=True, check=False)
    if install_res.stdout:
        print(install_res.stdout[-4000:])
    if install_res.returncode != 0:
        if install_res.stderr:
            print(install_res.stderr[-4000:])
        raise RuntimeError(f'uv pip install failed with code {install_res.returncode}')

    recheck = subprocess.run(probe, text=True, capture_output=True, check=False)
    recheck_out = (recheck.stdout or '').strip()
    print('[deps] Recheck output:', recheck_out)
    if 'MISSING=' in recheck_out and recheck_out.split('MISSING=', 1)[1].strip():
        raise RuntimeError(f'Modules still missing in uv env: {recheck_out}')
    print('[deps] uv environment dependencies are ready.')
else:
    print('[deps] uv environment already has required modules.')

[deps] Probe output: MISSING=torch,transformers,peft,trl,datasets,accelerate,bitsandbytes
Running: uv pip install --python .venv/bin/python torch transformers peft trl datasets accelerate bitsandbytes wrapt
[deps] Recheck output: MISSING=
[deps] uv environment dependencies are ready.


In [10]:
# Run full HF north-star pipeline (base ms, ft ms, delta, matrix).
LABEL_PREFIX = 'northstar-hf-kaggle-qwen25-3b'
CMD = [
    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',
    '--base-planner-config', 'configs/planners/hf_qwen2_5_3b_base.json',
    '--ft-planner-config', 'configs/planners/hf_qwen2_5_3b_ft.json',
    '--label-prefix', LABEL_PREFIX,
]

print('Running:', ' '.join(CMD))
started = time.time()
proc = subprocess.Popen(CMD, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
elapsed = time.time() - started
print(f'[northstar] Total elapsed: {elapsed:.1f}s')
if rc != 0:
    raise RuntimeError(f'North-star run failed with code {rc}')

Running: uv run python scripts/run_northstar_hf.py --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix northstar-hf-kaggle-qwen25-3b
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config configs/baseline.json --workload workloads/sample_tasks.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label northstar-hf-kaggle-qwen25-3b-base-ms

Loading weights: 100%|██████████| 434/434 [00:19<00:00, 21.83it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/northstar-hf-kaggle-qwen25-3b-base-ms/multi_seed.json
Wrote multi-seed summary: runs/northstar-hf-kaggle-qwen25-3b-base-ms/multi_seed_summary.md
[northstar] Stage finished in 40.2s
[northstar] Running: uv run tcrb multi-seed --config configs/baseline.json --workload workloads/sample_tasks.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_ft.json --label northstar-hf-

## Artifact Index

This section only lists generated artifact locations for quick navigation and reporting.

In [ ]:
label = 'northstar-hf-kaggle-qwen25-3b'
paths = [
    REPO_ROOT / 'runs' / f'{label}-base-ms' / 'multi_seed.json',
    REPO_ROOT / 'runs' / f'{label}-ft-ms' / 'multi_seed.json',
    REPO_ROOT / 'runs' / f'{label}-delta' / 'delta-ms.json',
    REPO_ROOT / 'runs' / f'{label}-matrix' / 'matrix.json',
]

print('Artifact files:')
for p in paths:
    print('-', p, 'exists=' + str(p.exists()))